# Single-Molecule Tracking Analysis. Telling molecular diffusional regimes apart.
This exercise is part of the *Técnicas Experimentales en Nanofísica y Biofísica* course of the *Máster Universitario en Física de la Materia Condensada y de los Sistemas Biológicos*


<div style="background-color: #f0f8ff; border: 2px solid #4682b4; padding: 10px;">
Jose Requejo Isidro
<br>Centro Nacional de Biotecnología (CNB, CSIC). Madrid

Ocober 2026
</div>

# Summary

In single-molecule tracking (SMT) the two numbers most people report are the diffusion coefficient $D$ and the anomalous exponent $\alpha$, defined through the mean squared displacement

$$\mathrm{MSD}(\tau) = 2\,d\,D\,\tau^{\alpha} \qquad (d=2 \;\Rightarrow\; \mathrm{MSD}=4D\tau^{\alpha})$$

$\alpha<1$ is called *subdiffusion*, $\alpha=1$ *normal (Brownian)*, $\alpha>1$ *superdiffusion*. 
These definitions are purely **descriptive**: they say *how* the MSD grows, not *why*. Many physical mechanisms produce the same $D$ and $\alpha$, and part of this exercise is learning to tell them apart. 

The point of this exercise is to explore how different physical mechanisms can give the same $(D, \alpha)$. In doing so, you will learn about additional observables that can help you tell the different diffusional regimes apart.

**Learning outcomes.** By the end you should be able to
1. Estimate $D$ and $\alpha$ from time-averaged MSDs and quantify their uncertainty and biases (fit range, trajectory length, localization error);
2. Explain why ensemble and time averages can disagree;
3. Detect heterogeneity (state switching, binding, trapping) that the MSD averages away, using jump-displacement distributions, the non-Gaussian parameter and windowed estimators;
4. Recognise the MSD signature of confinement and extract a confinement size;
5. Classify unlabelled trajectories into one of the five AnDi-2 models and justify the call.

### The AnDi challenges
The AnDi challenges are community competitions to benchmark methods for analysing single trajectories. The 1st challenge (2020) used trajectories from theoretical models (CTRW, fBm, Lévy walk, ATTM, scaled Brownian motion) and asked to infer the anomalous exponent α, classify the underlying model and locate a change in diffusive behaviour. The 2nd challenge (2024) used phenomenological, biologically motivated models and asked to detect changes of motion within trajectories and report $D$, α and the type of motion of each segment.

The 2nd Anomalous Diffusion (AnDi) Challenge provides simulators for five phenomenological scenarios that are common in biology.

https://github.com/AnDiChallenge
https://andichallenge.github.io/andi_datasets

**Phenomenological models** (`andi_datasets.models_phenom`):

| Model | Physics | Key parameters |
|---|---|---|
| `single_state` | one fractional-Brownian state | `Ds`, `alphas` |
| `multi_state` | Markov switching between states with different $D$/$\alpha$ | transition matrix `M` |
| `immobile_traps` | free diffusion + immobilisation at fixed traps | `Nt`, `r`, `Pb`, `Pu` |
| `dimerization` | molecules bind when they meet and move together, slower | `Pb`, `Pu`, `L` |
| `confinement` | diffusion inside compartments with semi-permeable walls | `r`, `Nc`, `trans` |

**Conventions.** Positions in pixels, time in frames ($\Delta t=1$). Trajectory arrays have shape `(T, N, 2)`. Ground-truth labels have shape `(T, N, 3)` with columns `[alpha, D, state]`; state codes: 0 immobile, 1 confined, 2 free, 3 directed.

## Basic concepts

**Jump displacement** It is the magnitude of the vector from one position to the next one. I.e. the jump displacement is the step distance $r$ at lag $\tau$ for $\tau=1$ 

**Ensemble-averaged and time-averaged MSD** We have $N$ trajectories $\mathbf{r}_i(t_k)$, $i = 1,\dots,N$, each recorded at $T$ frames $t_k = k\,\Delta t$, $k = 0,\dots,T-1$. Lags are $\tau = n\,\Delta t$ (here $\Delta t = 1$ frame).

**Ensemble-averaged MSD (EA-MSD)**: average over *particles* at a fixed time, measured from the start of the trajectory:

$$\langle \mathbf{r}^2(\tau)\rangle = \frac{1}{N}\sum_{i=1}^{N} \big|\mathbf{r}_i(\tau) - \mathbf{r}_i(0)\big|^2$$

Each particle contributes **one** displacement per lag, so precision is set by $N$. *Code:* `eamsd(trajs, max_lag)`.

**Time-averaged MSD (TA-MSD)**: average over *time* along a single trajectory, using every window of length $\tau$:

$$\overline{\delta^2_i}(\tau) = \frac{1}{T-n}\sum_{k=0}^{T-n-1} \big|\mathbf{r}_i(t_k + \tau) - \mathbf{r}_i(t_k)\big|^2$$

Each trajectory gives its own curve, built from $T-n$ **overlapping** displacements, so it becomes noisy at long lags, where few windows remain. *Code:* `tamsd(traj, max_lag)`.

**Mean TA-MSD**: the TA-MSD averaged over the $N$ trajectories:

$$\big\langle \overline{\delta^2}(\tau)\big\rangle = \frac{1}{N}\sum_{i=1}^{N} \overline{\delta^2_i}(\tau)$$

**Ergodicity.** A process is *ergodic in the MSD sense* if, for long trajectories, the time average of a single trajectory equals the ensemble average:

$$\lim_{T\to\infty} \overline{\delta^2_i}(\tau) = \langle \mathbf{r}^2(\tau)\rangle \quad \text{for every } i .$$

**Localization error** adds the same constant to both: with static noise of standard deviation $\sigma$ per coordinate, $\mathrm{MSD}_\text{measured}(\tau) = \mathrm{MSD}(\tau) + 4\sigma^2$ in 2D.

## Setting up

<div style="background-color: #f0f8ff; border: 2px solid #4682b4; padding: 10px;">
<strong>If using Kaggle:</strong> 

1. Uncomment the code in the cell below this one.
2. Run the cell. It will output an error due to conflicting `numpy` versions. Do not panic.
3. **Restart the kernel** in the three vertical dots menu $\vdots$ and proceed as usual. This will reload the `numpy` library.
</div>

In [ ]:
# !pip install andi-datasets   # Required if you use kaggle

<div style="background-color: #f0f8ff; border: 2px solid #4682b4; padding: 10px;">
<strong>If using Kaggle:</strong> 

1. Upload the dataset `mystery_trajectories.npz`. To do so: click Upload -> New dataset -> Name it -> Create
2. Uncomment the code in the cell below this one.
</div>

In [ ]:
# import glob, shutil
# for f in glob.glob("/kaggle/input/**/mystery_trajectories.npz", recursive=True):
#     shutil.copy(f, ".")

### Helper functions:
- `add_noise`: Adds static localization error: Gaussian noise of per-axis std sigma .
- `fit_all`: Per-trajectory (D, alpha) from TA-MSD fits. Returns arrays D, alpha. 
- `fit_msd`: Fit MSD = 4 D tau^alpha (+ 4 sigma^2) in log-log space on the first n_fit lags.
- `eamsd`: Ensemble-averaged MSD relative to t=0. trajs shape (T, N, 2).
- `tamsd`: Time-averaged MSD of one trajectory, shape (T, 2). Returns lags, msd.
- `nongaussian_parameter`: Computes the 2D non-Gaussian parameter alpha_2 = <r^4> / (2 <r^2>^2) - 1  (0 for a Gaussian).
- `windowed_D`: Local diffusion coefficient in a sliding window (assumes alpha=1)

In [ ]:
"""Helper functions for the AnDi-2 diffusion-modes exercise.

Conventions (AnDi 2 phenomenological models, 2D):
    MSD(tau) = 4 * D * tau**alpha      (tau in frames, positions in pixels)
    labels[..., 0] = alpha, labels[..., 1] = D, labels[..., 2] = state
    state codes: 0 = immobile, 1 = confined, 2 = free, 3 = directed
"""
import numpy as np


def tamsd(traj, max_lag):
    """Time-averaged MSD of one trajectory, shape (T, 2). Returns lags, msd."""
    lags = np.arange(1, max_lag + 1)
    msd = np.array([np.nanmean(np.sum((traj[l:] - traj[:-l]) ** 2, axis=1)) for l in lags])
    return lags, msd


def eamsd(trajs, max_lag):
    """Ensemble-averaged MSD relative to t=0. trajs shape (T, N, 2)."""
    lags = np.arange(1, max_lag + 1)
    disp = trajs[lags] - trajs[0][None]
    return lags, np.nanmean(np.sum(disp ** 2, axis=-1), axis=1)


def fit_msd(lags, msd, n_fit=None, sigma=0.0):
    """Fit MSD = 4 D tau^alpha (+ 4 sigma^2) in log-log space on the first n_fit lags.

    If sigma > 0 the known static localization error is subtracted first.
    Returns (D, alpha).
    """
    if n_fit is None:
        n_fit = len(lags)
    y = msd[:n_fit] - 4 * sigma ** 2
    x = lags[:n_fit]
    ok = y > 0
    if ok.sum() < 2:          # e.g. an immobile segment below the noise floor
        return np.nan, np.nan
    alpha, logc = np.polyfit(np.log(x[ok]), np.log(y[ok]), 1)
    return np.exp(logc) / 4, alpha


def fit_all(trajs, max_lag, n_fit, sigma=0.0):
    """Per-trajectory (D, alpha) from TA-MSD fits. Returns arrays D, alpha."""
    out = np.array([fit_msd(*tamsd(trajs[:, i], max_lag), n_fit=n_fit, sigma=sigma)
                    for i in range(trajs.shape[1])])
    return out[:, 0], out[:, 1]


def step_sizes(trajs, lag=1):
    """All displacement lengths at a given lag, flattened."""
    d = trajs[lag:] - trajs[:-lag]
    r = np.sqrt(np.sum(d ** 2, axis=-1)).ravel()
    return r[np.isfinite(r)]


def nongaussian_parameter(trajs, lag=1):
    """2D non-Gaussian parameter alpha_2 = <r^4> / (2 <r^2>^2) - 1  (0 for a Gaussian)."""
    r = step_sizes(trajs, lag)
    return np.mean(r ** 4) / (2 * np.mean(r ** 2) ** 2) - 1


def windowed_D(traj, w=10):
    """Local diffusion coefficient from squared steps in a sliding window (assumes alpha=1)."""
    sq = np.sum(np.diff(traj, axis=0) ** 2, axis=1)
    kernel = np.ones(w) / w
    return np.convolve(sq, kernel, mode="same") / 4


def add_noise(trajs, sigma, rng=None):
    """Static localization error: Gaussian noise of std sigma per coordinate."""
    rng = np.random.default_rng(rng)
    return trajs + rng.normal(0, sigma, trajs.shape)


def slow_segment_alpha(traj, thr, sigma=0.0, w=10, min_len=30):
    """alpha of the longest run whose windowed D is below thr (nan if none / unmeasurable)."""
    s = np.r_[windowed_D(traj, w) < thr, False]
    best, start = (0, 0), None
    for j, v in enumerate(s):
        if v and start is None:
            start = j
        if not v and start is not None:
            if j - start > best[1] - best[0]:
                best = (start, j)
            start = None
    if best[1] - best[0] < min_len:
        return np.nan
    lags, msd = tamsd(traj[best[0]:best[1] + 1], 8)
    return fit_msd(lags, msd, n_fit=4, sigma=sigma)[1]


### Model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from andi_datasets.models_phenom import models_phenom
# from diffusion_tools import (tamsd, eamsd, fit_msd, fit_all, step_sizes,
#                              nongaussian_parameter, windowed_D, add_noise,
#                              slow_segment_alpha)

np.random.seed(2026)         # andi_datasets uses NumPy's global RNG
# MODELS is a class that handles the generation of trajectories from different theoretical models.
MODELS = models_phenom()
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

---
## 1. Single-state anomalous diffusion. Meaning of $\alpha$ and ways to measure it

**Goal**: To learn to measure $D$ and $\alpha$ from the MSD, and how fit range and trajectory length limit their precision.

*Note*: The models will sample $D$ from a normal distribution centred in the first element and with the variance indicated by the second element in `Ds=[mean, variance]`. 
Same with $\alpha$: `alphas=[mean, var]`

***Advanced topic: Fractional Brownian motion (fBm)***

The introduction defined $D$ and $\alpha$ through the MSD alone. In ordinary Brownian motion each step is independent of the previous ones. Fractional Brownian motion (fBm) is a generalisation of ordinary Brownian motion in which successive steps are correlated by the Hurst coefficient (H). In `andi_datasets` each state is simulated as fBm with the $D$ and $\alpha$ you specify.

- α = 0.5 (H = 0.25): subdiffusive fBm. Successive steps are anticorrelated, so the molecule tends to reverse direction.
- α = 1 (H = 0.5): ordinary Brownian motion, which is fBm with uncorrelated steps.
- α = 1.5 (H = 0.75): superdiffusive fBm. Successive steps are positively correlated, so the motion is persistent.

**Subdiffusive fBm** describes motion in a viscoelastic environment: the surroundings push back elastically, so a displacement tends to be undone by the next one. It is a standard model for chromatin loci, tracers in crowded cytoplasm and some membrane proteins.

**Superdiffusive fBm** describes persistent motion, for example under active transport.

A molecule undergoing fBm may show  anomalous diffusion ($\alpha \neq 1$), but fBm is not the only process responsible for $\alpha \neq 1$. Other processes give the same $\alpha$ with different properties. For instance, negative step correlations also arise from confinement, because a molecule reverses direction at the walls. The MSD tells them apart: fBm gives a straight line on log–log axes, whereas confinement bends over into a plateau (Part 5).


**1.1** Simulate `N=200` trajectories of length `T=200` with `single_state` for $\alpha\in\{0.5, 1.0, 1.5\}$ and $D=1$. Plot five example trajectories of each and describe what you see.

`single_state`: https://andichallenge.github.io/andi_datasets/lib_nbs/models_phenom.html#single-state-model 

In [ ]:
T, N = 200, 200
alphas_true = [0.5, 1.0, 1.5]
ss = {a: MODELS.single_state(N=N, T=T, Ds=[1, 0], alphas=[a, 0])[0] for a in alphas_true}

fig, axs = plt.subplots(1, 3, figsize=(12, 4))
for ax, a in zip(axs, alphas_true):
    for i in range(5):
        x = ss[a][:, i] - ss[a][0, i]
        ax.plot(*x.T, lw=0.8)
    ax.set_title(f"α = {a}"); ax.set_aspect("equal"); ax.set_xlabel("x (px)")
axs[0].set_ylabel("y (px)")
plt.tight_layout()

**1.2** For every trajectory compute the time-averaged MSD up to lag 20 and fit $D$ and $\alpha$ on the first 5 lags. Plot the histograms of $\alpha$ and $D$ and the mean TA-MSD curves on log–log axes together with the theoretical $4D\tau^\alpha$. What is the difference between the three TA-MSD curves?

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for a in alphas_true:
    D, al = fit_all(ss[a], max_lag=20, n_fit=5)
    ax[0].hist(al, bins=30, alpha=0.6, label=f"α={a}")
    ax[1].hist(D, bins=30, alpha=0.6)
    msds = np.array([tamsd(ss[a][:, i], 50)[1] for i in range(N)])
    lags = np.arange(1, 51)
    ax[2].loglog(lags, msds.mean(0), "o", ms=3, label=f"α={a}")
    ax[2].loglog(lags, 4 * lags**a, "k--", lw=0.8)
    print(f"α_true={a}:  α̂ = {al.mean():.3f} ± {al.std():.3f},  D̂ = {D.mean():.3f} ± {D.std():.3f}")
ax[0].set_xlabel("fitted α"); ax[1].set_xlabel("fitted D (px²/frame)")
ax[2].set_xlabel("lag τ (frames)"); ax[2].set_ylabel("⟨TA-MSD⟩")
ax[0].legend(); plt.tight_layout()

**1.3 Bias and variance of the estimated anomalous exponent ($\hat\alpha$) .** For $\alpha=1$ repeat the fit with `n_fit` ∈ {2, 3, 5, 10, 20, 50, 100} (use `max_lag = n_fit`) and with trajectory lengths `T` ∈ {20, 50, 100, 200, 500} (fixed `n_fit=5`). Plot mean ± SD of $\hat\alpha$ against each variable.

*Note*: `n_fit` number of points used to fit $D$ and $\alpha$ ; `T` is the length of the track (used to compute the jump displacements)

To answer the following questions you will have to recall how the TA-MSD is computed.

*Questions*
- (a) Why does the scatter of $\hat\alpha$ grow when you fit to long lags?
- (b) How long must a trajectory be for a single-trajectory $\hat\alpha$ to distinguish 0.8 from 1.0 at ~2σ?
- (c) Why is $\hat\alpha$ noisier when only very short trajectories are available?

To answer question (c) you may have to repeat the $T$ sweep for α=1.5 and α=0.5 for completeness

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
nfits = [2, 3, 5, 10, 20, 50, 100]
res = [fit_all(ss[1.0], max_lag=n, n_fit=n)[1] for n in nfits]
ax[0].errorbar(nfits, [r.mean() for r in res], [r.std() for r in res], fmt="o-", capsize=3)
# ax[0].set_xscale("log"); 
ax[0].axhline(1, c="k", ls="--"); ax[0].set_xlabel("n_fit (lags)"); ax[0].set_ylabel("α̂")
ax[0].set_title(f"α = 1");

Ts = [20, 50, 100, 200, 500]
res_T = []
for TT in Ts:
    t = MODELS.single_state(N=200, T=TT, Ds=[1, 0], alphas=[1, 0])[0]
    res_T.append(fit_all(t, max_lag=5, n_fit=5)[1])
ax[1].errorbar(Ts, [r.mean() for r in res_T], [r.std() for r in res_T], fmt="o-", capsize=3)
# ax[1].set_xscale("log"); 
ax[1].axhline(1, c="k", ls="--"); ax[1].set_xlabel("trajectory length T"); ax[1].set_ylabel("α̂")
ax[1].set_title(f"α = 1");
plt.tight_layout()
for TT, r in zip(Ts, res_T):
    print(f"T={TT:4d}: σ(α̂) = {r.std():.3f}")

In [ ]:

fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))

Ts = [20, 50, 100, 200, 500]
alphas_true=[0.5, 1.5]
for ax, a in zip(axs, alphas_true):
    res_T = []
    for TT in Ts:
        t = MODELS.single_state(N=200, T=TT, Ds=[1, 0], alphas=[a, 0])[0]
        res_T.append(fit_all(t, max_lag=5, n_fit=5)[1])
    ax.errorbar(Ts, [r.mean() for r in res_T], [r.std() for r in res_T], fmt="o-", capsize=3)
    ax.axhline(a, c="k", ls="--"); ax.set_xlabel("trajectory length T"); ax.set_ylabel("α̂")
    # ax.set_xscale("log")
    ax.set_title(f"α = {a}");

plt.tight_layout()


**1.4 Ensemble vs time average.** Compute the ensemble-averaged MSD and compare with the mean TA-MSD for $\alpha=0.5$ and $1.5$. Then simulate `single_state` with a broad *population* of diffusion coefficients, `Ds=[1, 0.3]` (mean, variance), $\alpha=1$.

- Are trajectories ergodic? 
- With the heterogeneous $D$ population, compare the distribution of single-trajectory $\hat D$ with the input, and compute the non-Gaussian parameter $\alpha_2$ of the pooled steps. What does a population of *individually Gaussian* trajectories do to the pooled jump displacement distribution? Can α₂ alone tell you whether heterogeneity is between or within trajectories?

**Advanced topic: The non-Gaussian parameter $\alpha_2$***

The MSD uses only the second moment of the displacements, $\langle r^2\rangle$. Very different processes can share the same $\langle r^2\rangle$ and therefore the same MSD. The fourth moment, $\langle r^4\rangle$, tells us whether the steps are Gaussian-distributed or come from a mixture of different behaviours.

Note that $\alpha_2$ is the standard symbol in the literature, but it has nothing to do with the anomalous exponent $\alpha$. $\alpha$ describes how the MSD grows with time; $\alpha_2$ describes the shape of the jump displacement distribution.


**Definition (2D).** For the jump displacements $r$, pooled over all trajectories: $\alpha_2(\tau) = \frac{\langle r^4\rangle}{2\,\langle r^2\rangle^2} - 1$

- $\alpha_2= 0$: the displacements are Gaussian, as in homogeneous Brownian motion or fBm.
- $\alpha_2 > 0$: the displacements distribution has a sharper peak and heavier tails than a Gaussian. Mixtures of different diffusivities produce this: particle-to-particle heterogeneity (1.4), state switching (Part 3), trapping (Part 4).
- $\alpha_2 < 0$: the tails are lighter than a Gaussian’s. Steps are more uniform in length, as in directed motion with a constant speed.

**Mixture of molecules with different diffusion coefficients:** If each trajectory is Brownian with its own $D$, averaging the Gaussian moments over the distribution of $D$ gives $\alpha_2 = \frac{\langle D^2\rangle}{\langle D\rangle^2} - 1 = \frac{\mathrm{Var}(D)}{\langle D\rangle^2}$

Importantly, a mixture of individually Gaussian trajectories is not Gaussian when pooled.

*Check with the simulations:* in Part 3, about half of the frames have $D=1$ and half $D=0.05$, so $\langle D\rangle \approx 0.525$, $\langle D^2\rangle \approx 0.501$ and $\alpha_2 \approx 0.501/0.525^2 - 1 \approx 0.8$, close to the measured value. For the traps (Part 4), a fraction $p$ of frames is immobile ($D=0$) and the rest has $D=1$, so the formula gives $\alpha_2 = p/(1-p)$, about 0.7 for $p \approx 0.4$. Compare both predictions with the values printed in Parts 3 and 4 (they vary slightly from run to run).

Note that fBm jump displacements are Gaussian-distributed at every lag, so sub- and superdiffusive fBm both give $\alpha_2 \approx 0$. **Use the MSD for $\alpha$ and $\alpha_2$ for heterogeneity.**


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, a in zip(axs[:2], [0.5, 1.5]):
    lags, ea = eamsd(ss[a], 100)
    ta = np.array([tamsd(ss[a][:, i], 100)[1] for i in range(N)]).mean(0)
    ax.loglog(lags, ea, label="EA-MSD"); ax.loglog(lags, ta, "--", label="⟨TA-MSD⟩")
    ax.set_title(f"α = {a}"); ax.set_xlabel("τ"); ax.legend()

het, lab = MODELS.single_state(N=400, T=200, Ds=[1, 0.3], alphas=[1, 0])
Dhat, _ = fit_all(het, 20, 5)
axs[2].hist(lab[0, :, 1], bins=30, alpha=0.5, density=True, label="true D")
axs[2].hist(Dhat, bins=30, alpha=0.5, density=True, label="fitted D")
axs[2].set_xlabel("D"); axs[2].legend(); plt.tight_layout()
print(f"non-Gaussian parameter: homogeneous {nongaussian_parameter(ss[1.0]):.3f}, heterogeneous {nongaussian_parameter(het):.3f}")

---
## 2. Localization error makes Brownian motion look subdiffusive

**Goal**: To show that measurement noise makes Brownian motion look subdiffusive, and how to correct it.

Real tracks carry static localization error $\sigma$, so $\mathrm{MSD}(\tau)=4D\tau^\alpha+4\sigma^2$.

**2.1** Simulate Brownian motion with a *slow* $D=0.1$ and add Gaussian noise with $\sigma=0.5$ px (`add_noise`). Fit $\alpha$ on the first 5 lags as before. Then refit using the noise-corrected model (`sigma=0.5` in `fit_all`).

In [ ]:
clean = MODELS.single_state(N=200, T=200, Ds=[0.1, 0], alphas=[1, 0])[0]
noisy = add_noise(clean, 0.5, rng=1)
for name, kw in [("naive", {"sigma": 0.0}), ("σ-corrected", {"sigma": 0.5})]:
    D, a = fit_all(noisy, max_lag=20, n_fit=5, **kw)
    print(f"{name:12s}: α̂ = {np.median(a):.2f},  D̂ = {np.median(D):.3f}   (truth α=1, D=0.1)")


**2.2** Plot $\hat\alpha$ against the dimensionless ratio $\sigma^2/(D\Delta t)$ for a range of $\sigma$. At what ratio does the naive estimate become useless?

In [ ]:
sigmas = np.array([0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.8])
a_naive = [np.median(fit_all(add_noise(clean, s, 2), 20, 5)[1]) for s in sigmas]
a_corr = [np.median(fit_all(add_noise(clean, s, 2), 20, 5, sigma=s)[1]) for s in sigmas]
ratio = sigmas**2 / 0.1
plt.figure(figsize=(5, 3.6))
plt.semilogx(ratio[1:], a_naive[1:], "o-", label="naive"); plt.semilogx(ratio[1:], a_corr[1:], "s-", label="σ-corrected")
plt.axhline(1, c="k", ls="--"); plt.xlabel("σ² / (DΔt)"); plt.ylabel("median α̂"); plt.legend(); plt.tight_layout()

**2.3** **Advanced topic:** How would you estimate $\sigma$ from data if you did not know it? 

Two common ways: the MSD intercept (Michalet 2010) or the negative covariance of successive steps (Vestergaard, 2014).

In [ ]:
# 2.3: estimate sigma from the step covariance  <Δx_k Δx_{k+1}> = -σ²  (for Brownian motion, per coordinate) # Vestergaard et al. 2014
dx = np.diff(noisy, axis=0)
cov = np.mean(dx[1:] * dx[:-1])
print(f"σ from step covariance: {np.sqrt(-cov):.3f}  (true 0.5)")


---
## 3. Diffusional heterogeneity *within* trajectories

**Goal**: To show that the MSD analysis hides switching between multiple states, and reveal it with the jump-displacement distribution, the non-Gaussian parameter and windowed $D(t)$ analysis.

**3.1** Simulate molecules that switch between a fast ($D_1=1$) and a slow ($D_2=0.05$) Brownian state with transition matrix `M = [[0.98, 0.02], [0.02, 0.98]]` (per frame). Fit $D$ and $\alpha$ per trajectory exactly as in Part 1.
- What $\hat\alpha$ do you get? Is the MSD able to tell you that two states exist?
- How does the fitted $\hat D$ relate to $D_1$, $D_2$ and the fraction of time spent in each state?

`multi_state`: https://andichallenge.github.io/andi_datasets/lib_nbs/models_phenom.html#dataset-generation-1

In [ ]:
M = [[0.98, 0.02], [0.02, 0.98]]
ms, ms_lab = MODELS.multi_state(N=200, T=200, M=M, Ds=[[1, 0], [0.05, 0]], alphas=[[1, 0], [1, 0]], L=None)
D, a = fit_all(ms, 20, 5)
p_slow = (ms_lab[..., 1] < 0.5).mean()
print(f"α̂ = {np.median(a):.2f};  D̂ = {np.median(D):.3f};  time-weighted D = {(1-p_slow)*1 + p_slow*0.05:.3f}  (slow fraction {p_slow:.2f})")

**3.2** Plot the histogram of jump distances $r$ and overlay (i) a single 2D Rayleigh $p(r)=\frac{r}{2D}e^{-r^2/4D}$ with the global $\hat D$ and (ii) a two-component Rayleigh mixture with the true parameters. Compute the non-Gaussian parameter.

In [ ]:
r = step_sizes(ms)
fig, ax = plt.subplots(1, 1, figsize=(5, 3.8))
bins = np.linspace(0, 6, 80); rr = 0.5 * (bins[1:] + bins[:-1])
ray = lambda r, D: r / (2 * D) * np.exp(-r**2 / (4 * D))
ax.hist(r, bins=bins, density=True, alpha=0.5, label="data")
ax.plot(rr, ray(rr, np.median(D)), label="single Rayleigh (global D̂)")
ax.plot(rr, (1 - p_slow) * ray(rr, 1) + p_slow * ray(rr, 0.05), "k--", label="2-state mixture (truth)")
ax.set_xlabel("step length r (px)"); ax.legend()
# ax.set_yscale("log"); ax.set_ylim(1e-4, 5);
plt.tight_layout()

print(f"non-Gaussian parameter α2 = {nongaussian_parameter(ms):.2f}")

**3.3** Compute the windowed $D(t)$ for a few trajectories and overlay the ground-truth $D(t)$ from the labels. Threshold $D(t)$ to segment the trajectories, estimate the dwell times in each state and from them the switching probabilities. Compare with `M`. What happens when the window size is either too small or too large?

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
for i, c in zip(range(3), ["C0", "C1", "C2"]):
    axs[i].semilogy(windowed_D(ms[:, i], 10), c=c, lw=1)
    axs[i].semilogy(ms_lab[1:, i, 1], c=c, ls="--", lw=1)
    axs[i].set_xlabel("frame"); axs[i].set_ylabel("D(t)  (solid: windowed, dashed: truth)")
plt.tight_layout()

# segmentation and dwell times
thr = np.sqrt(1 * 0.05)                      # geometric midpoint between the two D's
dwell = {0: [], 1: []}
acc = []
for i in range(ms.shape[1]):
    s = (windowed_D(ms[:, i], 10) < thr).astype(int)  # 1 = slow
    acc.append(np.mean(s == (ms_lab[1:, i, 1] < 0.5)))
    edges = np.flatnonzero(np.diff(s)) + 1
    for seg in np.split(s, edges)[1:-1]:      # drop censored first/last segments
        dwell[seg[0]].append(len(seg))
for k, name in [(0, "fast"), (1, "slow")]:
    tau = np.mean(dwell[k]); print(f"{name}: mean dwell {tau:.1f} frames -> p_switch ≈ {1/tau:.3f} (true 0.02)")
print(f"frame-wise segmentation accuracy: {np.mean(acc):.2f}")

---
## 4. Interactions: traps and dimerization

**Goal**: To distinguish interaction-driven slow states from intrinsic switching, using immobility, fixed binding sites and concentration dependence.

Both the `immobile_traps` and `dimerization` models create a second, slower state, but **through interactions in space** rather than due to intrinsic origins (such as a molecule changing its D or α on its own, with a fixed probability per frame, independent of where it is and of the other molecules).

**4.1 Traps.** Using `immobile_traps` simulate free Brownian molecules ($D=1$) in a box `L=100` with `Nt=150` traps of radius `r=1`, `Pb=1` (binding probability), `Pu=0.05` (unbinding probability). Compare against `multi_state` from Part 3 using: fitted $\alpha$ and $D$, EA-MSD over 100 lags, jump distance histogram, and the fraction of exactly-immobile frames (state 0 in the labels).
- What is the distinctive signature of a trap in a single trajectory? (Look at a trajectory that gets trapped; plot its coordinates $x(t)$ and $y(t)$.)
- Repeat with `Nt` ∈ {25, 50, 100, 150, 300}. How does the bound fraction depend on trap density?

`immobile_traps`: https://andichallenge.github.io/andi_datasets/lib_nbs/models_phenom.html#immobile-traps 

In [ ]:
tr, tr_lab = MODELS.immobile_traps(N=200, T=200, L=100, r=1, Pu=0.05, Pb=1, Ds=[1, 0], alphas=[1, 0], Nt=150)
D, a = fit_all(tr, max_lag=20, n_fit=5)
print(f"traps: α̂ = {np.median(a):.2f}, D̂ = {np.median(D):.3f}, immobile frames = {(tr_lab[..., 2] == 0).mean():.2f}, α2 = {nongaussian_parameter(tr):.2f}")

fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
for t, name in [(ms, "multi_state"), (tr, "immobile_traps")]:
    lags, ea = eamsd(t, 100); axs[0].loglog(lags, ea / lags, label=name)
    axs[1].hist(step_sizes(t), bins=np.linspace(0, 6, 80), density=True, histtype="step", label=name)
axs[0].set_xlabel("τ"); axs[0].set_ylabel("EA-MSD / τ  (flat = normal)"); axs[0].legend()
axs[1].set_xlabel("Jump distance r"); axs[1].set_ylabel("Probability density")
axs[1].set_title("Jump displacements");

axs[1].legend()
#axs[1].set_yscale("log"); 

i = np.argmax(((tr_lab[..., 2] == 0).sum(0) > 30) & ((tr_lab[..., 2] == 2).sum(0) > 30))
axs[2].plot(tr[:, i, 0] - tr[0, i, 0], label="x"); axs[2].plot(tr[:, i, 1] - tr[0, i, 1], label="y")
axs[2].fill_between(range(200), *axs[2].get_ylim(), where=tr_lab[:, i, 2] == 0, color="grey", alpha=0.2, label="trapped")
axs[2].set_xlabel("frame"); axs[2].set_ylabel("Position");
axs[2].set_title("Coordinates");
axs[2].legend(); plt.tight_layout()

Nts = [25, 50, 100, 150, 300]
bound = [(MODELS.immobile_traps(N=200, T=200, L=100, r=1, Pu=0.05, Pb=1, Ds=[1, 0], alphas=[1, 0], Nt=n)[1][..., 2] == 0).mean() for n in Nts]
for n, b in zip(Nts, bound):
    print(f"Nt={n:3d}: bound fraction {b:.2f}")

**4.2 Dimerization.** Simulate molecule dimerization with `L=100, r=1, Pb=0.5, Pu=0.05`, monomer state $(D,\alpha)=(1,1)$ and dimer state $(0.1, 0.7)$.
- Fit per-trajectory $D$, $\alpha$ and plot them as a 2D scatter coloured by the fraction of time each molecule was dimerised.
- Now halve and double the box size `L` (i.e. change the molecule density). Which parameter of the measured behaviour changes, and why is this a hallmark of an *interaction* (bimolecular) rather than an intrinsic switch?

`dimerization`: https://andichallenge.github.io/andi_datasets/lib_nbs/models_phenom.html#time-evolution 

In [ ]:
def dimer_run(L):
    t, lab = MODELS.dimerization(N=200, T=200, L=L, r=1, Pu=0.05, Pb=0.5, Ds=[[1, 0], [0.1, 0]], alphas=[[1, 0], [0.7, 0]])
    D, a = fit_all(t, 20, 5)
    fdim = (lab[..., 1] < 0.5).mean(0)
    return t, lab, D, a, fdim

di, di_lab, D, a, fdim = dimer_run(100)
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
sc = axs[0].scatter(D, a, c=fdim, cmap="viridis", s=12); plt.colorbar(sc, ax=axs[0], label="fraction of time dimerised")
axs[0].set_xlabel("D̂"); axs[0].set_ylabel("α̂")
# axs[0].set_xscale("log"); 

for L in [50, 70, 100, 140, 200]:
    _, lab, D_, a_, f_ = dimer_run(L)
    print(f"L={L:3d} (density {200/L**2:.4f}/px²): dimer fraction {f_.mean():.2f}, median D̂ {np.median(D_):.3f}, median α̂ {np.median(a_):.2f}")
    axs[1].scatter(200 / L**2, f_.mean(), c="C0")
axs[1].set_xlabel("molecule density (1/px²)"); axs[1].set_ylabel("dimer fraction"); plt.tight_layout()
# axs[1].set_xscale("log"); 

---
## 5. Confinement: subdiffusion without anomalous motion

**Goal**: To show that geometry alone may result in apparent subdiffusion, recognisable by an MSD plateau that gives the compartment size.

**5.1** Simulate molecule confined diffusion with *Brownian motion both inside and outside* 60 circular compartments, $D=1$, $\alpha=1$, compartment radius `r=4`, `Nc=60` compartments in `L=100`, wall transmission `trans=0.01` (the wall transmission is the probability that a molecule crosses the wall when it reaches it):
- Fit $\alpha$ from the first 5 lags and from lags 20–100. What do you conclude about "anomalous" diffusion?
- Plot the TA-MSD of the trajectories that stay confined for the whole time. Show that it saturates and relate the plateau to $r$. (For a uniform distribution in a disk of radius $R$, compute $\langle |\mathbf{x}-\mathbf{y}|^2\rangle$.)
- Estimate the time to reach the plateau and relate it to $R^2/D$.

`confinement`: https://andichallenge.github.io/andi_datasets/lib_nbs/models_phenom.html#dataset-generation-2 

In [ ]:
co, co_lab = MODELS.confinement(N=200, T=200, L=100, Ds=[[1, 0], [1, 0]], alphas=[[1, 0], [1, 0]], r=4, Nc=60, trans=0.01)
lags = np.arange(1, 101)
msds = np.array([tamsd(co[:, i], 100)[1] for i in range(co.shape[1])])
a_short = [fit_msd(lags, m_, n_fit=5)[1] for m_ in msds]
a_long = [fit_msd(lags[19:], m_[19:])[1] for m_ in msds]
print(f"α̂(lags 1-5) = {np.median(a_short):.2f},   α̂(lags 20-100) = {np.median(a_long):.2f}   (model α = 1 everywhere)")

inside = (co_lab[..., 2] == 1).all(0)
plat = msds[inside][:, 50:].mean()
print(f"{inside.sum()} always-confined trajectories; plateau = {plat:.1f} px²  ->  R = sqrt(plateau) = {np.sqrt(plat):.2f} (true r = 4)")

fig, axs = plt.subplots(1, 1, figsize=(5, 3.8))
axs.loglog(lags, msds[inside].T, c="grey", alpha=0.2, lw=0.6)
axs.loglog(lags, msds[inside].mean(0), "k", lw=2, label="mean, confined")
axs.loglog(lags, 4 * lags, "r--", label="free 4Dτ"); axs.axhline(16, c="b", ls=":", label="R² = 16")
axs.set_ylim(1, 1e3); axs.set_xlabel("τ"); axs.set_ylabel("MSD");
axs.legend()

**5.2** Increase the wall transmittance `trans` to 0.1 and 0.5. What happens to the long-time MSD? Define and estimate a *macroscopic* (long-time) diffusion coefficient $D_\text{macro}$ and compare with $D_\text{micro}=1$ (hop diffusion, Kusumi et al.).

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(5, 3.8))
for trans in [0.01, 0.1, 0.5]:
    t, _ = MODELS.confinement(N=200, T=200, L=100, Ds=[[1, 0], [1, 0]], alphas=[[1, 0], [1, 0]], r=4, Nc=60, trans=trans)
    l_, ea = eamsd(t, 150)
    Dmacro = np.polyfit(l_[50:], ea[50:], 1)[0] / 4
    axs.plot(l_, ea / (4 * l_), label=f"trans={trans}: D_macro≈{Dmacro:.2f}")
axs.axhline(1, c="k", ls="--"); axs.set_xlabel("τ"); axs.set_ylabel("EA-MSD / 4τ  = D_app(τ)"); axs.legend()
plt.tight_layout()

---
## 6. Blind challenge: identify the model

`mystery_trajectories.npz` contains five datasets **A–E** (150 trajectories × 200 frames each, localization noise σ = 0.1 px, positions shifted to start at the origin). Each one was generated with exactly one of `single_state`, `multi_state`, `immobile_traps`, `dimerization`, `confinement`, all different, with parameters you have not seen.

For each dataset, compute at least:
1. single-trajectory $\hat D$, $\hat\alpha$ (short-lag fit, noise-corrected) and their spread;
2. $\hat\alpha$ from a long-lag fit and the shape of the TA-MSD (straight line? plateau?);
3. the non-Gaussian parameter of the pooled steps;
4. windowed $D(t)$ traces: are there switches, and is the slow state *immobile* (at the noise floor $\approx\sigma^2$) or merely slow? Fit $\alpha$ on the slow segments only (`slow_segment_alpha`): is the slow state Brownian or subdiffusive?

Fill in the table and **justify each assignment in two sentences**. Then state which pair was hardest to separate and what *additional experiment* (not analysis) you would do to separate them in the lab.

| Dataset | $\hat D$ | $\hat\alpha$ (short) | $\hat\alpha$ (long) | $\alpha_2$ | Switching? slow state immobile? | $\hat\alpha$ of slow segments | Model | Justification |
|---|---|---|---|---|---|---|---|---|
| A | | | | | | | | |
| B | | | | | | | | |
| C | | | | | | | | |
| D | | | | | | | | |
| E | | | | | | | | |

In [ ]:
data = np.load("mystery_trajectories.npz")
sigma = 0.1
summary = {}
fig, axs = plt.subplots(2, 5, figsize=(18, 6.5))
for j, k in enumerate("ABCDE"):
    t = data[k].astype(float)
    D, a = fit_all(t, 20, 5, sigma=sigma)
    lags = np.arange(1, 101)
    msds = np.array([tamsd(t[:, i], 100)[1] for i in range(t.shape[1])])
    a_long = np.median([fit_msd(lags[19:], m_[19:])[1] for m_ in msds])
    a2 = nongaussian_parameter(t)
    wD = np.array([windowed_D(t[:, i], 10) for i in range(t.shape[1])])
    slow_floor = np.mean(wD < 3 * sigma**2 / 2)        # windows at the noise floor (immobile)
    cv_within = np.median(wD.std(1) / wD.mean(1))      # switching within trajectories
    a_slow = np.nanmedian([slow_segment_alpha(t[:, i], thr=0.2, sigma=sigma) for i in range(t.shape[1])])
    summary[k] = dict(D=np.median(D), a_short=np.median(a), a_sd=a.std(), a_long=a_long, a2=a2,
                      cv_within=cv_within, frac_immobile=slow_floor, a_slow_segments=a_slow)
    axs[0, j].loglog(lags, msds.T, c="grey", alpha=0.1, lw=0.5); axs[0, j].loglog(lags, msds.mean(0), "k")
    axs[0, j].loglog(lags, 4 * np.median(D) * lags**np.median(a), "r--"); axs[0, j].set_title(k)
    for i in range(4): axs[1, j].semilogy(wD[i], lw=0.8)
    axs[1, j].set_ylim(1e-3, 10); axs[1, j].set_xlabel("frame")
axs[0, 0].set_ylabel("TA-MSD"); axs[1, 0].set_ylabel("windowed D(t)"); plt.tight_layout()

import pandas as pd
pd.DataFrame(summary).T.round(3)

## Further reading: 

- *andi_datasets* documentation: https://andichallenge.github.io/andi_datasets

- Kusumi et al. 2005. *Paradigm Shift of the Plasma Membrane Concept from the Two-Dimensional Continuum Fluid to the Partitioned Fluid: High-Speed Single-Molecule Tracking of Membrane Molecules*. Annu. Rev. Biophys. 34:351–378. https://doi.org/10.1146/annurev.biophys.34.040204.144637
  <br>High-speed tracking shows that the plasma membrane is partitioned into compartments by the actin membrane skeleton ("fences") and the transmembrane proteins anchored to it ("pickets"), so molecules diffuse freely inside a compartment and occasionally hop to the next one (hop diffusion).

- Metzler et al. 2014. *Anomalous diffusion models and their properties: non-stationarity, non-ergodicity, and ageing at the centenary of single particle tracking*. Phys. Chem. Chem. Phys. 16:24128–24164. https://doi.org/10.1039/c4cp03465a
  <br>Review of the main anomalous-diffusion models (CTRW, fBm, scaled Brownian motion and others) that can give the same MSD exponent but differ in stationarity, ergodicity and ageing, with tools to tell them apart.

- Muñoz-Gil et al. 2021. *Objective comparison of methods to decode anomalous diffusion*. Nat. Commun. 12:6253. https://doi.org/10.1038/s41467-021-26320-w
  <br>Results of the 1st AnDi challenge, benchmarking methods that infer α, classify the theoretical diffusion model and segment single trajectories; machine-learning methods performed best.

- Muñoz-Gil et al. 2025. *Quantitative evaluation of methods to analyze motion changes in single-particle experiments*. Nat. Commun. 16:6749. https://doi.org/10.1038/s41467-025-61949-x
  <br>Results of the 2nd AnDi challenge, benchmarking methods that detect changes in motion within trajectories and videos simulated with biologically motivated models, and that report D, α and the type of motion of each segment.

- Michalet 2010. *Mean square displacement analysis of single-particle trajectories with localization error: Brownian motion in an isotropic medium*. Phys. Rev. E 82:041914. https://doi.org/10.1103/PhysRevE.82.041914
  <br>Shows how localization error and finite trajectory length affect MSD fits for Brownian motion, and gives the optimal number of MSD points to fit for estimating D and the localization error.

- Vestergaard et al. 2014. *Optimal estimation of diffusion coefficients from single-particle trajectories*. Phys. Rev. E 89:022726. https://doi.org/10.1103/PhysRevE.89.022726
  <br>Introduces the covariance-based estimator (CVE), which uses the covariance of successive steps to estimate D and the localization error, and shows that it beats MSD fitting.

## Questions:
### 1. Single-state fractional Brownian motion: meaning of $\alpha$ and ways to measure it:
**1.1** Simulate `N=200` trajectories of length `T=200` with `single_state` for $\alpha\in\{0.5, 1.0, 1.5\}$ and $D=1$. Plot five example trajectories of each and describe what you see.

**1.2** For every trajectory compute the time-averaged MSD up to lag 20 and fit $D$ and $\alpha$ on the first 5 lags. Plot the histograms of $\alpha$ and $D$ and the mean TA-MSD curves on log–log axes together with the theoretical $4D\tau^\alpha$. What is the difference between the three TA-MSD curves?

**1.3** Bias and variance of the estimated anomalous exponent ($\hat\alpha$). For $\alpha=1$ repeat the fit with `n_fit` ∈ {2, 3, 5, 10, 20, 50, 100} (use `max_lag = n_fit`) and with trajectory lengths `T` ∈ {20, 50, 100, 200, 500} (fixed `n_fit=5`). Plot mean ± SD of $\hat\alpha$ against each variable.
- (a) Why does the scatter of $\hat\alpha$ grow when you fit to long lags?
- (b) How long must a trajectory be for a single-trajectory $\hat\alpha$ to distinguish 0.8 from 1.0 at ~2σ?
- (c) Why is $\hat\alpha$ noisier when only very short trajectories are available?

### 2. Localization error makes Brownian motion look subdiffusive
**2.1** Simulate Brownian motion with a *slow* $D=0.1$ and add Gaussian noise with $\sigma=0.5$ px (`add_noise`). Fit $\alpha$ on the first 5 lags as before. Then refit using the noise-corrected model (`sigma=0.5` in `fit_all`).

**2.2** Plot $\hat\alpha$ against the dimensionless ratio $\sigma^2/(D\Delta t)$ for a range of $\sigma$. At what ratio does the naive estimate become useless?

### 3. Diffusional heterogeneity *within* trajectories
**3.1** Simulate molecules that switch between a fast ($D_1=1$) and a slow ($D_2=0.05$) Brownian state with transition matrix `M = [[0.98, 0.02], [0.02, 0.98]]` (per frame). Fit $D$ and $\alpha$ per trajectory exactly as in Part 1.
- What $\hat\alpha$ do you get? Is the MSD able to tell you that two states exist?
- How does the fitted $\hat D$ relate to $D_1$, $D_2$ and the fraction of time spent in each state?

**3.2** Plot the histogram of jump distances $r$ and overlay (i) a single 2D Rayleigh $p(r)=\frac{r}{2D}e^{-r^2/4D}$ with the global $\hat D$ and (ii) a two-component Rayleigh mixture with the true parameters. Compute the non-Gaussian parameter.

**3.3** Compute the windowed $D(t)$ for a few trajectories and overlay the ground-truth $D(t)$ from the labels. Threshold $D(t)$ to segment the trajectories, estimate the dwell times in each state and from them the switching probabilities. Compare with `M`. What happens when the window size is either too small or too large?

### 4. Interactions: traps and dimerization`
**4.1 Traps.** Simulate free Brownian molecules ($D=1$) in a box `L=100` with `Nt=150` traps of radius `r=1`, `Pb=1`, `Pu=0.05`. Compare against `multi_state` from Part 3 using: fitted $\alpha$ and $D$, EA-MSD over 100 lags, jump-distance histogram, and the fraction of exactly-immobile frames (state 0 in the labels).
- What is the distinctive signature of a trap in a single trajectory? (Look at a trajectory that gets trapped; plot its coordinates $x(t)$ and $y(t)$.)
- Repeat with `Nt` ∈ {25, 50, 100, 150, 300}. How does the bound fraction depend on trap density?

**4.2 Dimerization.** Simulate `dimerization` with `L=100, r=1, Pb=0.5, Pu=0.05`, monomer state $(D,\alpha)=(1,1)$ and dimer state $(0.1, 0.7)$.
- Fit per-trajectory $D$, $\alpha$ and plot them as a 2D scatter coloured by the fraction of time each molecule was dimerised.
- Now halve and double the box size `L` (i.e. change the molecule density). Which parameter of the measured behaviour changes, and why is this a hallmark of an *interaction* (bimolecular) rather than an intrinsic switch?

### 5. Confinement: subdiffusion without anomalous motion
**5.1** Simulate `confinement` with *Brownian motion both inside and outside* 60 circular compartments, $D=1$, $\alpha=1$, compartment radius `r=4`, `Nc=60` compartments in `L=100`, wall transmission `trans=0.01` (the wall transmission is the probability that a molecule crosses the wall when it reaches it):
- Fit $\alpha$ from the first 5 lags and from lags 20–100. What do you conclude about "anomalous" diffusion?
- Plot the TA-MSD of the trajectories that stay confined for the whole time. Show that it saturates and relate the plateau to $r$. (For a uniform distribution in a disk of radius $R$, compute $\langle |\mathbf{x}-\mathbf{y}|^2\rangle$.)
- Estimate the time to reach the plateau and relate it to $R^2/D$.

**5.2** Increase the wall transmittance `trans` to 0.1 and 0.5. What happens to the long-time MSD? Define and estimate a *macroscopic* (long-time) diffusion coefficient $D_\text{macro}$ and compare with $D_\text{micro}=1$ (hop diffusion, Kusumi et al.).

### 6. Blind challenge: identify the model

Fill in the table and **justify each assignment in two sentences**. Then state which pair was hardest to separate and what *additional experiment* (not analysis) you would do to separate them in the lab.

| Dataset | $\hat D$ | $\hat\alpha$ (short) | $\hat\alpha$ (long) | $\alpha_2$ | Switching? slow state immobile? | $\hat\alpha$ of slow segments | Model | Justification |
|---|---|---|---|---|---|---|---|---|
| A | | | | | | | | |
| B | | | | | | | | |
| C | | | | | | | | |
| D | | | | | | | | |
| E | | | | | | | | |